In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

#import personnal tools
import sys
sys.path.append('../tools/')
from info import *
from imports import *
from tools_generic import *
from events import *

# Load files

In [ ]:
site_list=["d17","d47","d85","dmc"]
file_start_date = '20241201'
file_end_date = '20260630'

golden_start_date="2025-01-01"
golden_end_date="2025-03-31"

In [ ]:
data = {}
daily_data = {}

In [ ]:
data = {}
data = load_flowcapt_data(data, site_list, file_start_date, file_end_date)
data = load_spc_data(data, site_list, file_start_date, file_end_date)
data = load_surf_data(data, site_list, file_start_date, file_end_date)
data = load_wind_data(data, site_list, file_start_date, file_end_date) # WIND loading includes the early WIND_BEGINNING files by default.
data = load_mrr_precip_fraction(
    data,
    site=["d17", "d47"],
    file_start_date=golden_start_date.replace("-", ""),
    file_end_date=golden_end_date.replace("-", ""),
    timestep="30min",
    dbz_threshold=1.0,
    altitude_range=(300, 1000),
    variable_name="precip_fraction_mrr",
)

In [ ]:
# Extract period of interest (golden month)
data = filter_datasets_golden(
    data, start_date=golden_start_date, end_date=golden_end_date
 )

In [ ]:
# create daily
daily_data = create_daily_data(data)

# Stats

In [ ]:
variable = 'FluxMean1'
compute_variable_stats(data, variable)

In [ ]:
var = "FluxMean1_flowcapt"
plot_binned_distribution(data, var, bin_number=30, min_value=0, max_value=200)


In [ ]:
var = "snowflux_spc"
plot_binned_distribution(data, var, bin_number=30, min_value=0, max_value=200)


# Basic plotting

## Time plots

In [ ]:
variables = ["wdir_wind", "wspd1_wind", 'precip_fraction_mrr']

plot_per_var_multiple_sites(
    sensor_datasets=data,
    variables=variables,
    sites=["d17", "d47", "d85"],
    ymin=[0, 0, 0],
    ymax=[360, 26, 1],
)


In [ ]:
data['d17']

In [ ]:
variables = ["FluxMean1_flowcapt", "FluxMean2_flowcapt", "snowflux_spc"]

plot_per_site_multiple_vars(
    data,
    variables,
    sites=["d17", "d47", "d85"],
    figsize=(15, 5),
    ymax=30,
)


## Scatters

In [ ]:
var1 = "wdir_wind"
var2 = "wspd1_wind"
site1 = "d17"
site2 = "d17"
plot_bivariate_scatter(
    data,
    var1=var1,
    var2=var2,
    site1=site1,
    site2=site2,
    show_corr=False,
    show_fit=False,
    min_val=[0, 0],
    max_val=[360, 26],
    figsize=(7, 6),
)


In [ ]:
var1 = "FluxMean2_flowcapt"
var2 = "snowflux_spc"
var3 = "wspd1_wind"
site1 = "d47"
site2 = "d47"
site3 = "d47"
plot_trivariate_scatter(
    data,
    var1=var1,
    var2=var2,
    var3=var3,
    site1=site1,
    site2=site2,
    site3=site3,
    min3=10,
    max3=20,
    max_val=[300, 300],
    figsize=(7, 6),
    show_oneone=True,
)


# Events

In [ ]:
detector = EventDetector(
    threshold=1.0,
    min_timesteps=12,
    buffer_timesteps=0,
)

sampled_data = create_resampled_data(data, "30min")
additional_variables = [
    "FluxMean1_flowcapt",
    "snowflux_spc",
    "wspd1_wind",
    "wspd2_wind",
    "wdir_wind",
    "Hagl_flowcapt",
    "T1_surf",
    "RH1_surf",
    'precip_fraction_mrr'
]
collection = detector.detect_events(
    sampled_data,
    variable="FluxMean2_flowcapt",
    additional_variables=additional_variables,
)
collection_d17 = collection.get_site("d17")
collection_d47 = collection.get_site("d47")
non_collection = detector.detect_non_events(
    sampled_data,
    variable="FluxMean2_flowcapt",
    additional_variables=additional_variables,
)
non_collection_d17 = non_collection.get_site("d17")
non_collection_d47 = non_collection.get_site("d47")


In [ ]:
collection_d17.to_catalog(variables=["FluxMean2"])#.head()

In [ ]:
non_collection_d17.to_catalog(variables=["FluxMean2"])#.head()

In [ ]:
collection_d47.to_catalog(variables=["FluxMean2"])

In [ ]:
print_duration_stats(collection)
print_duration_stats(collection_d17)
print_duration_stats(collection_d47)

In [ ]:
print_integrated_flux_stats(collection)
print_integrated_flux_stats(collection_d17)
print_integrated_flux_stats(collection_d47)

In [ ]:
plot_diurnal_start_distribution(collection_d17)

### Time series

In [ ]:
plot_event_collection_traces(
    collection=collection_d17,
    variable="wspd1_wind",
    align_to="start_time",
    time_unit="h",
    cmap_name="viridis",
    alpha=1,
    linewidth=0.8,
    show_mean=False,
)


In [ ]:
plot_event_collection_traces(
    collection=non_collection_d17,
    variable="FluxMean2_flowcapt",
    align_to="start_time",
    time_unit="h",
    cmap_name="viridis",
    alpha=1,
    linewidth=0.7,
)


In [ ]:
plot_event_collection_traces(
    collection=collection_d47,
    variable="wspd1_wind",
    align_to="start_time",
    time_unit="h",
    cmap_name="viridis",
    alpha=1,
    linewidth=0.7,
    show_mean=False,
)


In [ ]:
varlist = ["FluxMean2_flowcapt", "wspd1_wind", "wdir_wind", 'precip_fraction_mrr']
plot_events_vs_nonevents_chronological(
    collection_d17,
    non_collection_d17,
    variables=varlist,
    figsize=(15, 15),
)
plot_events_vs_nonevents_chronological(
    collection_d47,
    non_collection_d47,
    variables=varlist,
    figsize=(15, 15),
)


In [ ]:
var1 = "wspd1_wind"
var2 = "FluxMean2_flowcapt"
plot_collection_bivariate_scatter(
    collection=collection,
    var1=var1,
    var2=var2,
    show_corr=False,
    show_fit=False,
)
plot_collection_bivariate_scatter(
    collection=non_collection,
    var1=var1,
    var2=var2,
    show_corr=False,
    show_fit=False,
)


### Composites

In [ ]:
composite = compute_event_composite(
    events=collection,
    variables=["FluxMean1_flowcapt", "FluxMean2_flowcapt", "snowflux_spc"],
    align_to="start_time",
    time_unit="h",
)
composite_d17 = compute_event_composite(
    events=collection_d17,
    variables=["FluxMean1_flowcapt", "FluxMean2_flowcapt", "snowflux_spc"],
    align_to="start_time",
    time_unit="h",
)
composite_d47 = compute_event_composite(
    events=collection_d47,
    variables=["FluxMean1_flowcapt", "FluxMean2_flowcapt", "snowflux_spc"],
    align_to="start_time",
    time_unit="h",
)


In [ ]:
plot_event_composite(
    composite_ds=composite,
    variables=["FluxMean1_flowcapt", "FluxMean2_flowcapt", "snowflux_spc", "wspd1_wind"],
    use_quantiles=True,
)


In [ ]:
plot_event_composite(
    composite_ds=composite_d17,
    variables=["FluxMean1_flowcapt", "FluxMean2_flowcapt", "snowflux_spc", "wspd1_wind"],
    use_quantiles=True,
)


In [ ]:
plot_event_composite(
    composite_ds=composite_d47,
    variables=["FluxMean1_flowcapt", "FluxMean2_flowcapt", "snowflux_spc", "wspd1_wind"],
    use_quantiles=True,
)
